# Study Reader: render read-aloud audio

Renders the books Study Reader sent to this Drive folder (**Render audio on Colab… → Send to Colab**) with Kokoro, the voice the app uses, so reading aloud plays at once with no waiting.

1. *Runtime → Change runtime type →* a GPU (a **T4** is plenty; an **L4** is faster).
2. *Runtime → Run all*, and allow access to Google Drive.

Audio is saved in parts as it goes, so if Colab disconnects, run it again and it carries on where it stopped. The app copies finished parts when you open the book (or with **Copy rendered audio**).

Study Reader writes this notebook to the folder each time you send a book, so don't edit it there: your changes would be overwritten.

In [ ]:
FOLDER = "/content/drive/MyDrive/study-reader-tts"  # the folder Study Reader sends books to, as Colab sees it
PART_SENTENCES = 300   # sentences per audio file
OPUS_QUALITY = 0.9     # libsndfile's Opus compression level: 0.9 is about 30 kbit/s, clear for speech

from google.colab import drive
drive.mount("/content/drive")
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
# Kokoro needs Python 3.12 or older, so it gets its own environment (Colab's Python may be newer).
!pip -q install uv
![ -x /content/kokoro-env/bin/python ] || uv venv -q --python 3.12 /content/kokoro-env
!uv pip install -q -p /content/kokoro-env/bin/python "kokoro>=0.9.4,<0.10" "misaki[en]" "soundfile>=0.13" "en_core_web_sm @ https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl"

In [ ]:
%%writefile /content/render_books.py
import glob, json, os, re, shutil, sys, tempfile, time
import numpy as np, soundfile as sf, torch
from kokoro import KPipeline

FOLDER, PART_SENTENCES, OPUS_QUALITY = sys.argv[1], int(sys.argv[2]), float(sys.argv[3])

RATE = 24000
TMP = tempfile.mkdtemp()
assert torch.cuda.is_available(), "No GPU: choose a GPU runtime (Runtime → Change runtime type)"
pipelines = {}

def pipeline(lang):
    if lang not in pipelines:
        pipelines[lang] = KPipeline(lang_code=lang, repo_id="hexgrad/Kokoro-82M", device="cuda")
    return pipelines[lang]

def speak(text, voice, speed):
    """Samples and Kokoro's timed tokens [text, start s, end s] for one sentence."""
    parts, tokens, t0 = [], [], 0.0
    for r in pipeline(voice[0])(text, voice=voice, speed=speed, split_pattern=None):
        if r.audio is None:
            continue
        audio = r.audio.detach().cpu().numpy().astype(np.float32)
        for t in r.tokens or []:
            timed = t.start_ts is not None and t.end_ts is not None
            tokens.append([t.text, t0 + t.start_ts if timed else None, t0 + t.end_ts if timed else None])
        parts.append(audio)
        t0 += len(audio) / RATE
    return (np.concatenate(parts) if parts else np.zeros(0, np.float32)), tokens

def rendered(audio_dir, speed):
    done = set()
    for path in glob.glob(f"{audio_dir}/part-*.json"):
        if os.path.exists(path[:-5] + ".opus"):
            done |= {k for k, v in json.load(open(path))["items"].items() if abs(v[0] - speed) < 0.01}
    return done

def render_book(book_dir):
    request = json.load(open(f"{book_dir}/request.json"))
    voice, speed, sentences = request["voice"], request["speed"], request["sentences"]
    audio_dir = f"{book_dir}/audio"
    os.makedirs(audio_dir, exist_ok=True)
    done = rendered(audio_dir, speed)
    todo = [(k, t) for k, t in sentences if k not in done]
    print(f"{request['title']}: {len(sentences) - len(todo):,} of {len(sentences):,} sentences already rendered")
    numbers = [int(re.search(r"part-(\d+)", p).group(1)) for p in glob.glob(f"{audio_dir}/part-*.json")]
    number = max(numbers, default=-1) + 1
    start, audio_seconds = time.time(), 0.0
    for i in range(0, len(todo), PART_SENTENCES):
        clips, items, pos = [], {}, 0
        for key, text in todo[i:i + PART_SENTENCES]:
            audio, tokens = speak(text, voice, speed)
            items[key] = [speed, pos, len(audio), tokens]
            clips.append(audio)
            pos += len(audio)
        audio = np.concatenate(clips) if clips else np.zeros(0, np.float32)
        name = f"part-{number:04d}"
        # Written locally, then copied: the audio first and its index last, so the app never
        # sees an index whose audio isn't there.
        sf.write(f"{TMP}/{name}.opus", audio, RATE, format="OGG", subtype="OPUS", compression_level=OPUS_QUALITY)
        shutil.copyfile(f"{TMP}/{name}.opus", f"{audio_dir}/{name}.opus")
        json.dump({"version": 1, "rate": RATE, "items": items}, open(f"{TMP}/{name}.json", "w"))
        shutil.copyfile(f"{TMP}/{name}.json", f"{audio_dir}/{name}.json")
        number += 1
        done |= set(items)
        audio_seconds += len(audio) / RATE
        took = time.time() - start
        json.dump({"rendered": len(done & {k for k, _ in sentences}), "total": len(sentences), "updated": time.time()},
                  open(f"{book_dir}/status.json", "w"))
        print(f"  {len(done):,} of {len(sentences):,} · {audio_seconds / 3600:.1f} h of speech in {took / 60:.1f} min "
              f"({audio_seconds / max(took, 1e-9):.0f}× real time)", flush=True)
    json.dump({"rendered": len(sentences), "total": len(sentences), "updated": time.time()},
              open(f"{book_dir}/status.json", "w"))
    # Parts made at another speed whose sentences all have audio at this one now are not needed.
    current = rendered(audio_dir, speed)
    for path in glob.glob(f"{audio_dir}/part-*.json"):
        items = json.load(open(path))["items"]
        if items and all(abs(v[0] - speed) >= 0.01 and k in current for k, v in items.items()):
            os.remove(path)
            os.remove(path[:-5] + ".opus")
            print(f"  removed {os.path.basename(path)[:-5]}, made at {next(iter(items.values()))[0]}×")

books = sorted(os.path.dirname(p) for p in glob.glob(f"{FOLDER}/books/*/request.json"))
if not books:
    print(f"No books in {FOLDER}/books: send one from Study Reader (Render audio on Colab… → Send to Colab).")
for book_dir in books:
    render_book(book_dir)
print("Done. Study Reader copies the audio when you open the book, or with Copy rendered audio.")

In [ ]:
!/content/kokoro-env/bin/python -W ignore -u /content/render_books.py "{FOLDER}" {PART_SENTENCES} {OPUS_QUALITY}